# 🎬 Pipeline de Edición de Video — Reels / TikTok
**Canal:** Duvan — IA aplicada a Marketing y Ecommerce

### Pipeline (5 fases):
1. ✂️ Cortar silencios
2. 📝 Transcribir con AssemblyAI
3. 💬 Subtítulos animados (WebM con alpha)
4. 🎨 Overlay visual: hook + keyword pills
5. 🎥 Composite final

---
**Instrucciones:**
1. Ejecuta la celda de **Setup** (tarda 1-2 minutos)
2. Monta tu **Google Drive**
3. Edita la celda de **Configuración** con tus datos
4. Ejecuta las fases en orden con `Shift+Enter`
5. Descarga el video final al terminar

In [ ]:
# ═══════════════════════════════════════════════════════
# CELDA 1 — SETUP (ejecutar primero, tarda ~1 min)
# ═══════════════════════════════════════════════════════
import subprocess, sys

print('Instalando ffmpeg...')
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)

print('Instalando paquetes Python...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'assemblyai', 'pillow', 'requests'], check=True)

print('Descargando fuente Inter...')
import requests
FONT_SEMIBOLD = '/tmp/Inter-SemiBold.ttf'
FONT_BOLD     = '/tmp/Inter-Bold.ttf'
FONT_EXTRABOLD = '/tmp/Inter-ExtraBold.ttf'
try:
    base = 'https://github.com/rsms/inter/raw/master/docs/font-files/'
    for fname, path in [('Inter-SemiBold.ttf', FONT_SEMIBOLD),
                         ('Inter-Bold.ttf', FONT_BOLD),
                         ('Inter-ExtraBold.ttf', FONT_EXTRABOLD)]:
        r = requests.get(base + fname, timeout=15)
        if r.status_code == 200:
            open(path, 'wb').write(r.content)
    print('✅ Fuente Inter descargada')
except Exception as e:
    FONT_SEMIBOLD = FONT_BOLD = FONT_EXTRABOLD = None
    print(f'⚠️  Inter no disponible, usando fuente del sistema: {e}')

print('\n✅ Setup completo. Continúa con la siguiente celda.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELDA 2 — MONTAR GOOGLE DRIVE
# ═══════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELDA 3 — CONFIGURACIÓN ← EDITAR ESTO
# ═══════════════════════════════════════════════════════

# --- TU API KEY DE ASSEMBLYAI ---
# Obtener en: https://www.assemblyai.com/app (es gratis)
ASSEMBLYAI_API_KEY = 'PON_TU_API_KEY_AQUI'

# --- RUTA AL VIDEO EN GOOGLE DRIVE ---
# Sube tu video a Drive y pon la ruta aquí
# Ejemplo: '/content/drive/MyDrive/Videos/mi_video.mp4'
VIDEO_INPUT = '/content/drive/MyDrive/mi_video.mp4'

# --- TEXTO DEL HOOK (aparece primeros 5 segundos) ---
# Máximo 3 líneas. Usa \n para saltos de línea.
HOOK_TEXT = 'La herramienta que está\ncambiando cómo se hace\nmarketing con IA'

# --- PARÁMETROS DEL PIPELINE ---
SILENCE_MIN_S     = 0.5    # duración mínima de silencio a cortar (segundos)
SILENCE_NOISE_DB  = -35    # umbral de silencio en dB (más negativo = más sensible)
WORDS_PER_CUE     = 2      # palabras por cue de subtítulos
LANGUAGE          = 'es'   # idioma del video

# --- DIRECTORIO DE SALIDA ---
OUTPUT_DIR = '/content/output'

# --- VERIFICACIÓN ---
import os
from pathlib import Path
os.makedirs(OUTPUT_DIR, exist_ok=True)
if not Path(VIDEO_INPUT).exists():
    print(f'⚠️  Video no encontrado: {VIDEO_INPUT}')
    print('   Verifica que el archivo está en Drive y la ruta es correcta.')
else:
    size_mb = Path(VIDEO_INPUT).stat().st_size / 1e6
    print(f'✅ Video encontrado: {Path(VIDEO_INPUT).name} ({size_mb:.1f} MB)')

if ASSEMBLYAI_API_KEY == 'PON_TU_API_KEY_AQUI':
    print('⚠️  Falta la API key de AssemblyAI. Edita esta celda.')
else:
    print('✅ Configuración lista')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELDA 4 — FUNCIONES AUXILIARES (no editar)
# ═══════════════════════════════════════════════════════
import json, re, subprocess
from pathlib import Path

def get_duration(video_path):
    r = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
         '-of', 'csv=p=0', str(video_path)],
        capture_output=True, text=True, check=True
    )
    return float(r.stdout.strip())

def get_dimensions(video_path):
    r = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-show_entries', 'stream=width,height',
         '-select_streams', 'v:0', '-of', 'csv=p=0', str(video_path)],
        capture_output=True, text=True, check=True
    )
    w, h = r.stdout.strip().split(',')
    return int(w), int(h)

def silence_cut(input_path, output_path, min_silence_s=0.5, noise_db=-35):
    """Fase 1: corta silencios y genera base limpia."""
    print(f'Detectando silencios en {Path(input_path).name}...')
    r = subprocess.run(
        ['ffmpeg', '-i', str(input_path),
         '-af', f'silencedetect=noise={noise_db}dB:d={min_silence_s}',
         '-f', 'null', '-'],
        capture_output=True, text=True
    )
    starts, ends = [], []
    for line in r.stderr.split('\n'):
        m = re.search(r'silence_start: ([\d.]+)', line)
        if m: starts.append(float(m.group(1)))
        m = re.search(r'silence_end: ([\d.]+)', line)
        if m: ends.append(float(m.group(1)))

    duration = get_duration(input_path)
    silences = list(zip(starts, ends[:len(starts)]))
    print(f'  {len(silences)} silencios detectados, duración total: {duration:.1f}s')

    keeps = []
    t = 0.0
    for s_start, s_end in sorted(silences):
        if s_start - t > 0.15:
            keeps.append({'start': round(t, 3), 'end': round(s_start, 3)})
        t = s_end
    if duration - t > 0.15:
        keeps.append({'start': round(t, 3), 'end': round(duration, 3)})

    output_duration = sum(k['end'] - k['start'] for k in keeps)
    print(f'  {len(keeps)} segmentos a mantener → {output_duration:.1f}s de output')

    seg_files = []
    for i, k in enumerate(keeps):
        seg_path = f'/tmp/seg_{i:03d}.mp4'
        subprocess.run([
            'ffmpeg', '-y', '-ss', str(k['start']), '-to', str(k['end']),
            '-i', str(input_path),
            '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '18',
            '-c:a', 'aac', '-b:a', '192k', seg_path
        ], capture_output=True, check=True)
        seg_files.append(seg_path)
        print(f'  segmento {i+1}/{len(keeps)}', end='\r')

    concat_list = '/tmp/concat_list.txt'
    Path(concat_list).write_text('\n'.join(f"file '{s}'" for s in seg_files))
    subprocess.run([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_list,
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
        '-c:a', 'aac', '-b:a', '192k', str(output_path)
    ], capture_output=True, check=True)

    final_dur = get_duration(output_path)
    saved = duration - final_dur
    print(f'\n✅ Base video: {output_path}')
    print(f'   {duration:.1f}s → {final_dur:.1f}s (cortados {saved:.1f}s de silencio)')
    return keeps, final_dur


def transcribe_assemblyai(video_path, api_key, language='es'):
    """Fase 2: transcribe con AssemblyAI word-level."""
    import assemblyai as aai
    aai.settings.api_key = api_key

    audio_path = '/tmp/audio_mono.wav'
    print('Extrayendo audio mono 16kHz...')
    subprocess.run([
        'ffmpeg', '-y', '-i', str(video_path),
        '-ac', '1', '-ar', '16000', '-sample_fmt', 's16', audio_path
    ], capture_output=True, check=True)

    print('Transcribiendo con AssemblyAI...')
    config = aai.TranscriptionConfig(
        language_code=language,
        speech_model=aai.SpeechModel.best
    )
    transcriber = aai.Transcriber()
    transcript = transcriber.transcribe(audio_path, config=config)

    if transcript.status == aai.TranscriptStatus.error:
        raise RuntimeError(f'AssemblyAI error: {transcript.error}')

    words = [{
        'text':  w.text,
        'start': w.start,
        'end':   w.end,
        'type':  'word'
    } for w in (transcript.words or [])]

    print(f'✅ Transcripción: {len(words)} palabras')
    print('  Primeras 5 palabras:')
    for w in words[:5]:
        print(f"    {w['start']/1000:.2f}s  '{w['text']}'")
    return {'words': words, 'text': transcript.text}


def build_subtitle_cues(words, words_per_cue=2):
    """Agrupa palabras del transcript en cues."""
    # timestamps en ms → convertir a segundos
    word_list = [{'text': w['text'],
                  'start_s': w['start'] / 1000,
                  'end_s':   w['end']   / 1000}
                 for w in words if w.get('type', 'word') == 'word']

    cues = []
    for i in range(0, len(word_list), words_per_cue):
        group = word_list[i:i + words_per_cue]
        start_s    = group[0]['start_s']
        next_start = word_list[i + words_per_cue]['start_s'] \
                     if i + words_per_cue < len(word_list) \
                     else group[-1]['end_s'] + 0.5
        duration_s = max(next_start - start_s, 0.2)
        cues.append({
            'start_s':    start_s,
            'duration_s': duration_s,
            'words':      [w['text'] for w in group]
        })
    return cues


KEYWORD_PILLS = {
    'claude':         'Claude',
    'claude code':    'Claude Code',
    'chatgpt':        'ChatGPT',
    'chat gpt':       'ChatGPT',
    'gpt':            'GPT',
    'meta ads':       'Meta ADs',
    'facebook ads':   'Facebook Ads',
    'fb ads':         'FB Ads',
    'instagram':      'Instagram',
    'tiktok':         'TikTok',
    'agente ia':      'Agente IA',
    'agente':         'Agente IA',
    'automatizacion': 'Automatización',
    'automatización': 'Automatización',
    'ugc':            'UGC',
    'ecommerce':      'Ecommerce',
    'whatsapp':       'WhatsApp',
    'notion':         'Notion',
    'make':           'Make.com',
    'n8n':            'n8n',
    'openai':         'OpenAI',
    'gemini':         'Gemini',
    'anthropic':      'Anthropic',
}

def detect_keyword_events(words):
    """Detecta keywords en el transcript y retorna eventos."""
    events = []
    for i, w in enumerate(words):
        word_lower = w['text'].lower().strip('.,!?¡¿:;"\'')
        next_lower = words[i+1]['text'].lower().strip('.,!?¡¿:;"\'')\
                     if i + 1 < len(words) else ''
        bigram = word_lower + ' ' + next_lower

        matched = None
        for kw, display in KEYWORD_PILLS.items():
            if ' ' in kw and kw in bigram:
                matched = display
                break
        if not matched:
            for kw, display in KEYWORD_PILLS.items():
                if ' ' not in kw and kw == word_lower:
                    matched = display
                    break
        if matched:
            events.append({'start_s': w['start']/1000, 'text': matched, 'duration_s': 2.0})

    # Deduplicar: misma keyword con gap mínimo de 4s
    deduped, last = [], {}
    for ev in sorted(events, key=lambda x: x['start_s']):
        last_t = last.get(ev['text'], -99)
        if ev['start_s'] - last_t > 4.0:
            deduped.append(ev)
            last[ev['text']] = ev['start_s']
    return deduped


print('✅ Funciones auxiliares cargadas')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELDA 5 — RENDERIZADOR DE SUBTÍTULOS Y OVERLAY (no editar)
# ═══════════════════════════════════════════════════════
from PIL import Image, ImageDraw, ImageFont
import math

def load_font(path, size):
    if path:
        try: return ImageFont.truetype(path, size)
        except: pass
    try: return ImageFont.truetype('/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf', size)
    except: return ImageFont.load_default()

def draw_pill(draw, text, x, y, font, bg_rgba, text_rgba, padding_h=10, padding_v=4, radius=5):
    bbox = draw.textbbox((0, 0), text, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    w = tw + 2 * padding_h
    h = th + 2 * padding_v
    draw.rounded_rectangle([x, y, x + w, y + h], radius=radius, fill=bg_rgba)
    draw.text((x + padding_h, y + padding_v + (h - 2*padding_v - th)//2),
              text, fill=text_rgba, font=font)
    return w, h

def alpha_at(t, start_s, duration_s, fade=0.12):
    end_s = start_s + duration_s
    if t < start_s or t > end_s: return 0.0
    fade = min(fade, duration_s * 0.3)
    if t - start_s < fade:  return (t - start_s) / fade
    if end_s - t    < fade:  return (end_s - t)   / fade
    return 1.0


def render_subtitles_webm(cues, output_path, video_duration,
                           fps=30, canvas_w=1080, canvas_h=1920,
                           font_size=28, bottom=385, gap=5):
    """Renderiza subtítulos karaoke como WebM VP9 con alpha."""
    font_semi = load_font(FONT_SEMIBOLD, font_size)
    total_frames = int(video_duration * fps) + 1

    cmd = [
        'ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
        '-s', f'{canvas_w}x{canvas_h}', '-pix_fmt', 'rgba',
        '-r', str(fps), '-i', 'pipe:0',
        '-c:v', 'libvpx-vp9', '-pix_fmt', 'yuva420p',
        '-b:v', '0', '-crf', '32', '-auto-alt-ref', '0',
        str(output_path)
    ]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)

    for fi in range(total_frames):
        t = fi / fps
        img = Image.new('RGBA', (canvas_w, canvas_h), (0, 0, 0, 0))
        draw = ImageDraw.Draw(img)

        active = next((c for c in cues
                       if c['start_s'] <= t < c['start_s'] + c['duration_s']), None)
        if active:
            a = alpha_at(t, active['start_s'], active['duration_s'])
            words = active['words']

            # Medir ancho total para centrar
            sizes = []
            for w in words:
                bb = draw.textbbox((0,0), w, font=font_semi)
                sizes.append((bb[2]-bb[0] + 20, bb[3]-bb[1] + 8))
            total_w = sum(s[0] for s in sizes) + gap*(len(words)-1)
            max_h   = max(s[1] for s in sizes)

            x = (canvas_w - total_w) // 2
            y = canvas_h - bottom - max_h

            for wi, (word, (ww, wh)) in enumerate(zip(words, sizes)):
                is_active = (wi == len(words) - 1)
                bg = (17, 17, 17, int(240*a)) if is_active else (255, 255, 255, int(235*a))
                fg = (255, 255, 255, int(255*a)) if is_active else (17, 17, 17, int(255*a))
                bb = draw.textbbox((0,0), word, font=font_semi)
                tw, th = bb[2]-bb[0], bb[3]-bb[1]
                draw.rounded_rectangle([x, y, x+ww, y+max_h], radius=5, fill=bg)
                draw.text((x+10, y+(max_h-th)//2), word, fill=fg, font=font_semi)
                x += ww + gap

        proc.stdin.write(img.tobytes())
        if fi % (fps*10) == 0:
            print(f'  subtítulos: frame {fi}/{total_frames} ({t:.0f}s/{video_duration:.0f}s)', end='\r')

    proc.stdin.close()
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError('ffmpeg falló al codificar subtítulos WebM')
    print(f'\n✅ Subtítulos WebM: {output_path}')


def render_overlay_webm(hook_text, keyword_events, output_path, video_duration,
                         fps=30, canvas_w=1080, canvas_h=1920):
    """Renderiza overlay (hook + keyword pills) como WebM VP9 con alpha."""
    font_hook    = load_font(FONT_BOLD,      32)
    font_keyword = load_font(FONT_EXTRABOLD, 38)
    total_frames = int(video_duration * fps) + 1

    # Pre-renderizar el hook box
    def make_hook_img(alpha_f):
        img = Image.new('RGBA', (canvas_w, canvas_h), (0, 0, 0, 0))
        if alpha_f <= 0: return img
        draw = ImageDraw.Draw(img)
        a = int(255 * alpha_f)
        lines = hook_text.replace('\\n', '\n').split('\n')

        line_sizes = []
        for line in lines:
            bb = draw.textbbox((0,0), line, font=font_hook)
            line_sizes.append((bb[2]-bb[0], bb[3]-bb[1]))

        max_lw = max(s[0] for s in line_sizes)
        line_h = max(s[1] for s in line_sizes)
        line_gap = 8
        total_h = len(lines)*line_h + (len(lines)-1)*line_gap
        pad_h, pad_v = 24, 16
        box_w = max_lw + 2*pad_h
        box_h = total_h + 2*pad_v
        box_x = (canvas_w - box_w) // 2
        box_y = 110

        draw.rounded_rectangle([box_x, box_y, box_x+box_w, box_y+box_h],
                                radius=12, fill=(255,255,255,int(242*alpha_f)))
        ty = box_y + pad_v
        for line, (lw, lh) in zip(lines, line_sizes):
            draw.text((box_x + (box_w-lw)//2, ty), line,
                      fill=(17,17,17,a), font=font_hook)
            ty += lh + line_gap
        return img

    cmd = [
        'ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
        '-s', f'{canvas_w}x{canvas_h}', '-pix_fmt', 'rgba',
        '-r', str(fps), '-i', 'pipe:0',
        '-c:v', 'libvpx-vp9', '-pix_fmt', 'yuva420p',
        '-b:v', '0', '-crf', '32', '-auto-alt-ref', '0',
        str(output_path)
    ]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)

    # Precalcular hook fade (0→0.5s = fade in, 4.7→5s = fade out)
    HOOK_START, HOOK_END = 0.0, 5.0

    for fi in range(total_frames):
        t = fi / fps
        img = Image.new('RGBA', (canvas_w, canvas_h), (0, 0, 0, 0))

        # Hook
        ha = alpha_at(t, HOOK_START, HOOK_END - HOOK_START, fade=0.4)
        if ha > 0:
            hook_layer = make_hook_img(ha)
            img = Image.alpha_composite(img, hook_layer)

        # Keyword pills
        draw = ImageDraw.Draw(img)
        for ev in keyword_events:
            a = alpha_at(t, ev['start_s'], ev['duration_s'], fade=0.25)
            if a <= 0: continue
            ai = int(255 * a)
            bb = draw.textbbox((0,0), ev['text'], font=font_keyword)
            tw, th = bb[2]-bb[0], bb[3]-bb[1]
            pw, ph = tw+40, th+20
            px = (canvas_w - pw) // 2
            py = int(canvas_h * 0.42) - ph//2
            draw.rounded_rectangle([px, py, px+pw, py+ph],
                                    radius=10, fill=(17,17,17,ai))
            draw.text((px+20, py+10), ev['text'], fill=(255,255,255,ai), font=font_keyword)

        proc.stdin.write(img.tobytes())
        if fi % (fps*10) == 0:
            print(f'  overlay: frame {fi}/{total_frames} ({t:.0f}s/{video_duration:.0f}s)', end='\r')

    proc.stdin.close()
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError('ffmpeg falló al codificar overlay WebM')
    print(f'\n✅ Overlay WebM: {output_path}')


print('✅ Renderizadores cargados')

In [ ]:
# ═══════════════════════════════════════════════════════
# FASE 1 — CORTAR SILENCIOS
# ═══════════════════════════════════════════════════════
BASE_VIDEO = Path(OUTPUT_DIR) / 'base_video.mp4'

keeps, base_duration = silence_cut(
    input_path   = VIDEO_INPUT,
    output_path  = BASE_VIDEO,
    min_silence_s = SILENCE_MIN_S,
    noise_db      = SILENCE_NOISE_DB
)

# Verificación visual — mostrar frame del segundo 3
check_path = str(Path(OUTPUT_DIR) / 'check_fase1.jpg')
subprocess.run(['ffmpeg', '-y', '-ss', '3', '-i', str(BASE_VIDEO),
                '-vframes', '1', '-q:v', '2', check_path], capture_output=True)

from IPython.display import Image as IPImage, display
print('\n📸 Frame de verificación (segundo 3 del base):')
display(IPImage(check_path, width=300))
print('\n✅ Fase 1 completa. Verifica que el video se ve bien antes de continuar.')

In [ ]:
# ═══════════════════════════════════════════════════════
# FASE 2 — TRANSCRIBIR CON ASSEMBLYAI
# ═══════════════════════════════════════════════════════
transcript_path = Path(OUTPUT_DIR) / 'transcript.json'

transcript = transcribe_assemblyai(
    video_path = BASE_VIDEO,
    api_key    = ASSEMBLYAI_API_KEY,
    language   = LANGUAGE
)

transcript_path.write_text(json.dumps(transcript, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\n💾 Transcript guardado: {transcript_path}')
print(f'   Total palabras: {len(transcript["words"])}')
print(f'\n📝 Primeras 10 palabras:')
for w in transcript['words'][:10]:
    print(f"   {w['start']/1000:.2f}s  '{w['text']}'")

In [ ]:
# ═══════════════════════════════════════════════════════
# FASE 3 — SUBTÍTULOS ANIMADOS (WebM con alpha)
# ═══════════════════════════════════════════════════════
subtitles_webm = Path(OUTPUT_DIR) / 'subtitles.webm'

cues = build_subtitle_cues(transcript['words'], words_per_cue=WORDS_PER_CUE)
print(f'📝 {len(cues)} cues generados')
print(f'   Primeros 3 cues:')
for c in cues[:3]:
    print(f"   {c['start_s']:.2f}s  {c['words']}")

print(f'\n🎬 Renderizando subtítulos WebM ({base_duration:.0f}s × 30fps = {int(base_duration*30)} frames)...')
render_subtitles_webm(
    cues          = cues,
    output_path   = subtitles_webm,
    video_duration = base_duration
)

# Verificar alpha
r = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-show_entries', 'stream=pix_fmt',
     '-of', 'default=noprint_wrappers=1', str(subtitles_webm)],
    capture_output=True, text=True
)
pix_fmt = r.stdout.strip()
if 'yuva420p' in pix_fmt:
    print(f'✅ Alpha confirmado: {pix_fmt}')
else:
    print(f'⚠️  pix_fmt inesperado: {pix_fmt} (esperado yuva420p)')

In [ ]:
# ═══════════════════════════════════════════════════════
# FASE 4 — OVERLAY VISUAL (hook + keyword pills)
# ═══════════════════════════════════════════════════════
overlay_webm = Path(OUTPUT_DIR) / 'overlay.webm'

keyword_events = detect_keyword_events(transcript['words'])
print(f'🎯 Keywords detectadas: {len(keyword_events)}')
for ev in keyword_events:
    print(f"   {ev['start_s']:.2f}s → '{ev['text']}'")

print(f'\n🎨 Renderizando overlay WebM...')
render_overlay_webm(
    hook_text      = HOOK_TEXT,
    keyword_events = keyword_events,
    output_path    = overlay_webm,
    video_duration = base_duration
)

# Verificar alpha
r = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-show_entries', 'stream=pix_fmt',
     '-of', 'default=noprint_wrappers=1', str(overlay_webm)],
    capture_output=True, text=True
)
pix_fmt = r.stdout.strip()
if 'yuva420p' in pix_fmt:
    print(f'✅ Alpha confirmado: {pix_fmt}')
else:
    print(f'⚠️  pix_fmt inesperado: {pix_fmt}')

In [ ]:
# ═══════════════════════════════════════════════════════
# FASE 5 — COMPOSITE FINAL
# ═══════════════════════════════════════════════════════
# base + subtítulos + overlay → final.mp4
# REGLA: -vcodec libvpx-vp9 ANTES de cada -i .webm
# REGLA: overlay=x=0:y=0 (canvas 1080×1920 = mismo tamaño que video)

final_output = Path(OUTPUT_DIR) / 'final_video.mp4'

print('🎥 Compositando video final...')
cmd = [
    'ffmpeg', '-y',
    '-i', str(BASE_VIDEO),
    '-vcodec', 'libvpx-vp9', '-i', str(subtitles_webm),
    '-vcodec', 'libvpx-vp9', '-i', str(overlay_webm),
    '-filter_complex',
    '[0:v][1:v]overlay=x=0:y=0:format=auto[v1];[v1][2:v]overlay=x=0:y=0:format=auto[vout]',
    '-map', '[vout]', '-map', '0:a',
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '18', '-pix_fmt', 'yuv420p',
    '-c:a', 'aac', '-b:a', '192k',
    '-movflags', '+faststart',
    str(final_output)
]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print('ERROR ffmpeg:')
    print(result.stderr[-2000:])
else:
    size_mb = Path(final_output).stat().st_size / 1e6
    print(f'✅ Video final: {final_output} ({size_mb:.1f} MB)')

    # Verificación visual — frames en segundo 2, 10, 30
    from IPython.display import display as ipy_display
    for t, label in [(2, 'hook'), (10, 'sin_hook'), (30, 'subtitulos')]:
        check = str(Path(OUTPUT_DIR) / f'check_{label}.jpg')
        subprocess.run(['ffmpeg', '-y', '-ss', str(t), '-i', str(final_output),
                        '-vframes', '1', '-q:v', '2', check], capture_output=True)
        if Path(check).exists():
            print(f'\n📸 Segundo {t} ({label}):')
            ipy_display(IPImage(check, width=300))

In [ ]:
# ═══════════════════════════════════════════════════════
# DESCARGAR RESULTADO
# ═══════════════════════════════════════════════════════
from google.colab import files

print(f'Descargando {Path(final_output).name}...')
files.download(str(final_output))

# Opcional: también copiar a Drive
import shutil
drive_out = '/content/drive/MyDrive/' + Path(final_output).name
shutil.copy(str(final_output), drive_out)
print(f'✅ También guardado en Drive: {drive_out}')

---
## 📌 Notas

- **AssemblyAI gratis:** 100 horas de audio/mes. Crear cuenta en assemblyai.com
- **Fuentes:** Se descarga Inter automáticamente. Si falla, usa Liberation Sans (similar a Arial)
- **Tiempo estimado:** ~3-8 min para un video de 90s (mayoría es transcripción + render WebM)
- **Si algo falla:** Cada fase guarda su output en `/content/output/`. Puedes reejecutar solo la fase que falló.
- **Ajustar hook text:** Editar `HOOK_TEXT` en la celda de Configuración y re-ejecutar desde Fase 4
- **Ajustar font size subtítulos:** Cambiar el parámetro `font_size=28` en la llamada `render_subtitles_webm()`